# 체인(Chain)이란?

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. 체이닝(Chaining) 개념을 이해하고 `PromptTemplate | Model | OutputParser` 구조를 설명할 수 있어요
2. 파이프(`|`) 연산자로 체인을 구성하고 `invoke()`로 실행할 수 있어요
3. `RunnableLambda`로 일반 Python 함수를 체인에 삽입할 수 있어요
4. 여러 LLM 호출을 연결하는 다단계 체인을 만들 수 있어요

## 사전 지식

- `01_LangChain Quick Start.ipynb`의 `invoke()`, `AIMessage` 개념
- Python 함수, 람다 표현식 기초

## 이전 노트북 연결

`01_LangChain Quick Start.ipynb`에서 모델에 직접 문자열이나 메시지 객체를 전달했어요. 이제 프롬프트 템플릿(Prompt Template)과 출력 파서(Output Parser)를 파이프 연산자로 연결해 재사용 가능한 체인을 만들어볼게요.

아래 다이어그램은 이 노트북에서 다룰 체인 흐름을 보여줘요.

```mermaid
flowchart LR
    A["딕셔너리 입력<br/>{topic: ...}"]:::input --> B["PromptTemplate<br/>템플릿 채우기"]:::process
    B --> C["ChatOpenAI<br/>LLM 호출"]:::process
    C --> D["StrOutputParser<br/>텍스트 추출"]:::process
    D --> E["RunnableLambda<br/>커스텀 변환"]:::process
    E --> F["최종 출력<br/>문자열 또는 딕셔너리"]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

체이닝은 여러 작업을 연속으로 연결해 실행하는 방식이에요. 한 작업의 결과가 다음 작업의 입력으로 이어지므로, 복잡한 워크플로우(Workflow)를 간결하게 표현할 수 있어요.

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

load_dotenv()

# 모델 초기화
# temperature 미지정 시 기본값 사용 (적당한 창의성)
model = ChatOpenAI(model="gpt-4o-mini")

## 1. 기본적인 체이닝: PromptTemplate + OutputParser

`PromptTemplate`과 `StrOutputParser(String Output Parser)`를 파이프(`|`) 연산자로 연결해 첫 번째 체인을 만들어볼게요.

- `PromptTemplate.from_template()`: `{변수명}` 형식의 플레이스홀더를 자동으로 인식해요
- `StrOutputParser`: `AIMessage`에서 `.content` 문자열만 추출해요
- 파이프 연산자: 왼쪽 출력이 오른쪽 입력으로 전달돼요

In [9]:
# ---------------------------------------------------
# PromptTemplate + Model + OutputParser 체인 구성하기
# ---------------------------------------------------
# ============================================================
# TODO: {question} 자리에 원하는 질문이 들어가도록 invoke()를 호출해요
# 힌트: invoke()에 딕셔너리 {"question": "..."} 형태로 전달해요
# 예상 결과: 질문에 대한 LLM 응답 문자열이 출력돼요
# ============================================================

# 1. PromptTemplate 생성
# from_template() : {변수명} 플레이스홀더를 자동으로 인식.
prompt_template = PromptTemplate.from_template(
    """다음 질문에 친절하게 대답해
    {question}"""
)

# 2. StrOutputParser 생성
# StrOutputParser : AIMessage에서 .content 문자열만 추출
output_parser = StrOutputParser()

# 3. 체인 생성
chain = prompt_template | model | output_parser

# 4. 체인 실행
result = chain.invoke({"question": "파이썬 리스트를 정렬하는 방법을 알려줘."})
print(result)

파이썬에서 리스트를 정렬하는 방법은 여러 가지가 있습니다. 가장 일반적인 두 가지 방법은 `sort()` 메서드와 `sorted()` 함수를 사용하는 것입니다. 각각의 방법을 설명해 드릴게요.

### 1. `sort()` 메서드
`sort()` 메서드는 리스트 객체에서 직접 호출하는 방법으로, 리스트를 정렬하고 그 결과를 리스트에 저장합니다. 이 메서드는 원본 리스트를 직접 변경하므로 반환값은 `None`입니다.

```python
# 예시
my_list = [5, 2, 9, 1, 5, 6]
my_list.sort()  # 리스트를 오름차순으로 정렬
print(my_list)  # 출력: [1, 2, 5, 5, 6, 9]
```

내림차순으로 정렬하고 싶다면 `reverse=True` 인자를 추가하면 됩니다.

```python
my_list.sort(reverse=True)  # 리스트를 내림차순으로 정렬
print(my_list)  # 출력: [9, 6, 5, 5, 2, 1]
```

### 2. `sorted()` 함수
`sorted()` 함수는 주어진 iterable을 정렬하여 새로운 리스트를 반환합니다. 원본 리스트는 변경되지 않습니다.

```python
# 예시
my_list = [5, 2, 9, 1, 5, 6]
sorted_list = sorted(my_list)  # 오름차순으로 정렬된 새로운 리스트 생성
print(sorted_list)  # 출력: [1, 2, 5, 5, 6, 9]
print(my_list)      # 원본 리스트는 변경되지 않음

# 내림차순으로 정렬할 때
sorted_list_desc = sorted(my_list, reverse=True)
print(sorted_list_desc)  # 출력: [9, 6, 5, 5, 2, 1]
```

### 정렬 기준 지정하기
두 방법 모두 `key` 인자를 사용하여 정렬 기준을 지정할 수 있습니다. 예를 들어, 문자열 리스트를 길이에 따라 정렬하고 싶으면 다음과 같이 할 수 있습

## 2. 여러 단계를 거치는 체이닝 (PromptTemplate + RunnableLambda)

PromptTemplate과 RunnableLambda를 사용하여 여러 단계를 파이프로 연결합니다.

> **한계점**: 아래 예제에서는 `add_topic_info` 함수 내에 `"파이썬"`이 하드코딩되어 있습니다. 파이프라인 중간에서 원본 입력값을 유지하려면 `RunnablePassthrough.assign()`이나 `RunnableParallel`을 사용해야 합니다. 이 패턴은 `04_Runnable_Basic.ipynb`에서 자세히 다룹니다.

In [12]:
# ---------------------------------------------------
# RunnableLambda로 중간 변환 함수 삽입하기
# ---------------------------------------------------

prompt_template = PromptTemplate.from_template(
    "{topic}에 대해 간단히 설명해줘"
)

# 사용자 커스텀 함수 정의 - 모델의 응답 결과를 커스터마이징
def format_output(data):
    topic = data['topic']
    response_text = data['response']

    return f"질문: {topic}\n 답변: {response_text}"

# 사용자의 입력 토픽을 유지하면서 체인 구성.
def create_formatted_chain():

    def add_topic_info(response_text):
        return {"topic": "파이썬", "response": response_text}

    return (
        prompt_template
        | model
        | StrOutputParser()
        | RunnableLambda(add_topic_info)
        | RunnableLambda(format_output)
    )

chain = create_formatted_chain()
print(chain.invoke({"topic": "파이썬"}))

질문: 파이썬
 답변: 파이썬(Python)은 높은 수준의 프로그래밍 언어로, 코드가 간결하고 읽기 쉽게 설계되었습니다. 1991년 귀도 반 로썸(Guido van Rossum)이 처음 개발했으며, 지금은 다양한 분야에서 널리 사용되고 있습니다. 

주요 특징은 다음과 같습니다:

1. **간결하고 명확한 문법**: 파이썬은 코드가 간단하고 이해하기 쉬워, 초보자들이 배우기에 적합합니다.

2. **다양한 라이브러리**: 수많은 내장 라이브러리와 패키지가 있어, 데이터 분석, 웹 개발, 머신러닝, 자동화 등 다양한 용도로 활용할 수 있습니다.

3. **플랫폼 독립성**: 윈도우, macOS, 리눅스 등 여러 운영체제에서 실행할 수 있습니다.

4. **객체 지향 프로그래밍**: 파이썬은 객체 지향 프로그래밍을 지원하며, 재사용 가능한 코드를 작성하는 데 유리합니다.

5. **대화형 프로그래밍**: 대화형 쉘을 지원하여, 코드를 실시간으로 실행하고 결과를 즉시 확인할 수 있습니다.

파이썬은 그 유연성과 강력한 기능 덕분에 많은 기업과 개발자들이 선호하는 언어입니다.


## 3. Runnable 체인을 함수로 만들기

앞서 `RunnableLambda`로 중간 변환을 삽입했어요. 체인 생성 로직 자체를 함수로 캡슐화하면 여러 곳에서 재사용하기가 편해요.

다음 셀에서 체인 생성 함수를 만들고 호출해볼게요.

In [13]:
# ---------------------------------------------------
# 체인 생성 로직을 함수로 캡슐화하기
# ---------------------------------------------------
# ============================================================
# TODO: create_explanation_chain()을 호출하고 다른 topic으로 실행해봐요
# 힌트: chain.invoke({"topic": "원하는 주제"}) 형태로 호출해요
# 예상 결과: 입력 주제에 대한 설명 문자열이 출력돼요
# ============================================================

def create_explaination_chain():
    prompt_template = PromptTemplate.from_template(
        "{topic}에 대해 간단히 설명해줘"
    )
    return prompt_template | model | StrOutputParser()

chain = create_explaination_chain()
result = chain.invoke({"topic": "머신러닝"})
print(result)

머신러닝(기계 학습)은 인공지능(AI)의 한 분야로, 컴퓨터가 데이터로부터 학습하고 예측 및 결정을 자동으로 수행할 수 있도록 하는 기술입니다. 머신러닝은 주어진 데이터에서 패턴을 인식하고, 그 패턴을 기반으로 새로운 데이터에 대한 예측이나 분류를 수행합니다.

머신러닝은 크게 세 가지 유형으로 나눌 수 있습니다:

1. **지도 학습(Supervised Learning)**: 입력 데이터와 해당하는 출력(정답) 데이터가 주어질 때, 모델이 이를 학습하여 새로운 데이터에 대한 예측을 수행합니다. 예를 들어, 이메일이 스팸인지 아닌지 분류하는 모델이 이에 해당합니다.

2. **비지도 학습(Unsupervised Learning)**: 출력 데이터가 주어지지 않고, 입력 데이터만을 이용하여 숨겨진 구조나 패턴을 찾는 방법입니다. 클러스터링(데이터 군집화)이나 차원 축소 등이 이 범주에 속합니다.

3. **강화 학습(Reinforcement Learning)**: 에이전트가 환경과 상호작용하면서 보상을 극대화하는 방향으로 학습하는 방식입니다. 주로 게임이나 로봇 제어와 같은 분야에서 사용됩니다.

머신러닝은 의료, 금융, 자율주행차, 자연어 처리 등 다양한 분야에서 활용되고 있습니다.


## 4. 파이프 연산자(`|`)를 사용한 체이닝

재사용 가능한 체인 함수를 만들어봤어요. 이제 파이프 연산자가 기존 방식과 어떻게 다른지 직접 비교해볼게요.

파이프 연산자를 사용하면 중간 변수 없이 `PromptTemplate | model | StrOutputParser()` 형태로 한 줄에 표현할 수 있어요. 데이터가 왼쪽에서 오른쪽으로 흐르는 방향이 코드에서 바로 보여요.

In [19]:
# ---------------------------------------------------
# 파이프 없이 각 단계를 수동으로 실행하는 방식 (비교용)
# ---------------------------------------------------
# 각 단계의 출력을 중간 변수에 담아 다음 단계로 전달해야 해요
prompt_template = PromptTemplate.from_template("사용자에 질문에 대해 친절하게 답변해 : {question}")
prompt_value = prompt_template.invoke({"question": "파이썬에 대해 간단히 알려줘."})

model_response = model.invoke(prompt_value)

output_parser = StrOutputParser()
print(output_parser.invoke(model_response))

파이썬(Python)은 1991년에 귀도 반 로섬(Guido van Rossum)이 만든 고급 프로그래밍 언어입니다. 파이썬은 간결하고 가독성이 높은 문법을 가지고 있어 초보자에게 배우기 쉬운 언어로 알려져 있습니다. 또한, 다양한 응용 분야에서 사용할 수 있는 유연성과 강력한 라이브러리를 제공합니다.

주요 특징은 다음과 같습니다:

1. **가독성**: 코드가 간결하고 명확하여 이해하기 쉽습니다.
2. **다양한 라이브러리**: 데이터 분석, 웹 개발, 인공지능 등 여러 분야에서 사용되는 많은 라이브러리가 있습니다. 예를 들어, NumPy(수치 계산), Pandas(데이터 분석), Django(웹 개발) 등이 있습니다.
3. **플랫폼 독립성**: 윈도우, macOS, 리눅스 등 다양한 운영체제에서 사용할 수 있습니다.
4. **객체 지향 프로그래밍**: 객체 지향 프로그래밍을 지원하여 코드 재사용성을 높일 수 있습니다.
5. **대화형**: 터미널이나 Jupyter Notebook과 같은 환경에서 대화형으로 코드를 실행해볼 수 있어 실험과 학습에 유용합니다.

파이썬은 웹 개발, 데이터 과학, 인공지능, 자동화 스크립트 작성 등 다양한 분야에서 광범위하게 사용되고 있습니다. 배우기 쉬우면서도 강력한 기능을 제공하기 때문에 많은 개발자와 기업들이 선호하는 언어입니다. 추가로 궁금한 점이 있다면 언제든지 물어보세요!


파이프 연산자를 사용하면 위의 코드를 더 간결하게 작성할 수 있습니다:


In [20]:
# ---------------------------------------------------
# 파이프 연산자로 체인을 한 줄에 표현하기
# ---------------------------------------------------
# ============================================================
# TODO: summary_prompt 변수를 정의하고 파이프로 chain을 구성해요
# 힌트: PromptTemplate.from_template("{text}") | model | StrOutputParser()
# 예상 결과: 입력 텍스트를 한 문장으로 요약한 결과가 출력돼요
# ============================================================

# 요약용 프롬프트
summary_prompt = PromptTemplate.from_template(
    """다음 문장을 한 문장으로 요약해줘:
    {text}"""
)

text = """파이썬은 배우기 쉽고 강력한 프로그래밍 언어로, 웹 개발, 데이터 분석, 인공지능 등 다양한 분야에서 활용됩니다."""

chain = summary_prompt | model | StrOutputParser()
print(chain.invoke({"text": text}))

파이썬은 쉽고 강력한 프로그래밍 언어로 웹 개발, 데이터 분석, 인공지능 등 다양한 분야에 사용됩니다.


### 파이프 연산자의 동작 원리

파이프 연산자(`|`)는 왼쪽의 Runnable 결과를 오른쪽 Runnable의 입력으로 전달합니다:
- `PromptTemplate | model` → 프롬프트 결과를 model에 전달
- `model | OutputParser` → model의 결과를 OutputParser에 전달
- `PromptTemplate | model | OutputParser` → 순차적으로 연결

데이터가 왼쪽에서 오른쪽으로 흐르는 것을 시각적으로 표현할 수 있어 코드가 더 읽기 쉬워집니다.

> **참고**: LangChain에서는 `__or__` 메서드를 오버라이드하여 `|` 연산자로 체인을 연결할 수 있어요. 이는 특정 Python 버전에 의존하지 않으며, LangChain이 지원하는 모든 Python 버전에서 동작합니다.


In [23]:
# ---------------------------------------------------
# RunnableLambda로 후처리 함수를 체인에 추가하기
# ---------------------------------------------------
prompt_template = PromptTemplate.from_template("{question}")

# 모델의 출력을 커스터마이징한다.
def format_output(response_text):
    return f"❤️❤️❤️❤️❤️❤️답변❤️❤️❤️❤️❤️❤️\n {response_text}"

chain = (
    prompt_template
    | model
    | StrOutputParser()
    | RunnableLambda(format_output)
)

print(chain.invoke("머신러닝이란 무엇인가요?"))

❤️❤️❤️❤️❤️❤️답변❤️❤️❤️❤️❤️❤️
 머신러닝(Machine Learning)은 인공지능(AI) 분야의 한 분야로, 컴퓨터가 명시적으로 프로그래밍되지 않고도 데이터에서 학습할 수 있도록 하는 기술입니다. 머신러닝의 주요 목표는 주어진 데이터에서 패턴이나 규칙을 찾아내고, 이를 바탕으로 새로운 데이터에 대한 예측이나 결정을 내리는 것입니다.

머신러닝은 크게 세 가지 주요 유형으로 나뉘어집니다:

1. **지도학습(Supervised Learning)**: 입력 데이터와 그에 대한 정답(label)이 주어졌을 때, 모델이 입력 데이터와 정답 간의 관계를 학습합니다. 이후 새로운 입력 데이터에 대해 예측을 수행합니다. 예를 들어, 이메일이 스팸인지 아닌지를 분류하는 것이 있습니다.

2. **비지도학습(Unsupervised Learning)**: 입력 데이터만 주어지고 정답이 주어지지 않을 때, 데이터의 구조나 패턴을 찾아내는 학습 방법입니다. 클러스터링이나 차원 축소 등이 여기에 해당합니다. 예를 들어, 고객 세그멘테이션이 있습니다.

3. **강화학습(Reinforcement Learning)**: 에이전트가 환경과 상호작용하면서 보상을 최대화하도록 학습하는 방법입니다. 에이전트는 행동을 선택하고 그에 따른 보상을 받으면서 최적의 전략을 찾아갑니다. 예를 들어, 게임 플레이어가 주어진 환경에서 승리를 목표로 하는 경우입니다.

머신러닝은 이미지 인식, 자연어 처리, 추천 시스템, 자율주행차 등 다양한 분야에서 활용되고 있습니다.


### 파이프 연산자의 장점

- **가독성**: 코드의 흐름이 왼쪽에서 오른쪽으로 자연스럽게 읽힙니다
  - `입력 → 처리1 → 처리2 → 출력` 형태로 데이터 흐름을 직관적으로 표현
- **간결성**: 중간 변수를 줄여 코드가 더 짧아집니다
- **연결성**: 여러 단계를 쉽게 연결할 수 있습니다
  - 새로운 단계를 추가할 때 `| 새로운단계`만 붙이면 됩니다


In [ ]:
# ---------------------------------------------------
# 파이프 사용 전·후 코드 비교하기
# ---------------------------------------------------
prompt_template = PromptTemplate.from_template("{topic}에 대해 설명해줘")
output_parser = StrOutputParser()

prompt_value = prompt_template.invoke({"topic": "딥러닝"})
response = model.invoke(prompt_value)
output = output_parser.invoke(response)
print("방법 1:", output[:50] + "...")

# 방법 2: 파이프 연산자 사용 (더 간결하고 읽기 쉬움)
# 데이터 흐름이 왼쪽→오른쪽으로 코드에서 그대로 드러나요
chain2 = prompt_template | model | output_parser
output2 = chain2.invoke({"topic": "딥러닝"})
print("방법 2:", output2[:50] + "...")

## 5. 여러 LLM 호출을 연결하는 체인

파이프 연산자의 장점을 확인했어요. 이제 한 LLM의 출력을 다음 LLM의 입력으로 연결하는 다단계 체인을 만들어볼게요.

첫 번째 체인이 설명을 생성하고, 두 번째 체인이 그 설명을 요약해요. 두 체인을 하나로 합칠 때 `RunnableLambda`로 데이터 형식(문자열 → 딕셔너리)을 변환하는 방법도 확인할 수 있어요.

In [ ]:
# ---------------------------------------------------
# 두 LLM 호출을 연결하는 다단계 체인 만들기
# ---------------------------------------------------
# ============================================================
# TODO: explanation_chain의 출력을 summary_chain의 입력으로 연결해요
# 힌트: RunnableLambda(lambda text: {"text": text})로 문자열→딕셔너리 변환
# 예상 결과: "파이썬"에 대한 설명을 생성하고, 그것을 다시 한 문장으로 요약해요
# ============================================================

# 1단계: 설명 생성 체인
explanation_prompt = PromptTemplate.from_template("{topic}에 대해 간단히 설명해줘")
explanation_chain = explanation_prompt | model | StrOutputParser()

# 2단계: 요약 생성 체인
# StrOutputParser 출력(문자열)을 {text}에 넣어 요약 요청
summary_prompt = PromptTemplate.from_template(
    "다음 내용을 한 문장으로 요약해줘:\n\n{text}"
)
summary_chain = summary_prompt | model | StrOutputParser()

# 수동 연결 — 첫 번째 체인의 결과를 직접 두 번째 체인에 전달
explanation = explanation_chain.invoke({"topic": "파이썬"})
print("첫 번째 응답:")
print(explanation)
print("\n" + "="*50 + "\n")

summary = summary_chain.invoke({"text": explanation})
print("두 번째 응답 (요약):")
print(summary)

# 자동 연결 — RunnableLambda로 체인 간 데이터 형식을 변환
# 핵심: explanation_chain은 문자열 반환, summary_chain은 {"text": ...} 딕셔너리를 기대
print("\n" + "="*50)
print("한 번에 실행 (RunnableLambda 사용):")
full_chain = (
    explanation_chain 
    | RunnableLambda(lambda text: {"text": text})  # 문자열 → 딕셔너리 변환
    | summary_chain
)
final_result = full_chain.invoke({"topic": "파이썬"})
print(final_result)

In [ ]:
# ---------------------------------------------------
# RunnableLambda로 요약 프롬프트를 문자열로 직접 생성하기
# ---------------------------------------------------

# 첫 번째 단계: 설명 생성
explanation_prompt = PromptTemplate.from_template("{topic}에 대해 간단히 설명해줘")

# 두 번째 단계: 요약 프롬프트를 문자열로 직접 만들기
# model은 딕셔너리뿐 아니라 문자열 입력도 받아요 → PromptTemplate 없이 연결 가능
def create_summary_prompt(explanation_text):
    """설명 텍스트를 받아 요약 프롬프트 문자열로 변환"""
    return f"다음 내용을 한 문장으로 요약해줘:\n\n{explanation_text}"

# 체인 흐름: {topic} → 설명 → 문자열 요약 프롬프트 → LLM → 문자열
chain = (
    explanation_prompt 
    | model 
    | StrOutputParser()
    | RunnableLambda(create_summary_prompt)  # 문자열 → 프롬프트 문자열 변환
    | model
    | StrOutputParser()
)

result = chain.invoke({"topic": "파이썬"})
print("최종 요약 결과:")
print(result)

## 6. OutputParser를 활용한 구조화된 출력

다단계 체인을 다뤄봤어요. 이번에는 LLM 응답을 딕셔너리 형태로 파싱하는 커스텀 파서를 만들어볼게요.

`StrOutputParser`는 단순 문자열만 반환해요. 더 복잡한 구조가 필요할 때는 `RunnableLambda`로 파싱 함수를 체인에 연결할 수 있어요.

> **실무 팁**: 프로덕션 환경에서는 LLM 출력 형식이 일정하지 않을 수 있어요. `PydanticOutputParser`나 `JsonOutputParser` 같은 견고한 파서를 사용하는 편이 안정적이에요. 이 내용은 이후 노트북에서 다룰 거예요.

In [ ]:
# ---------------------------------------------------
# RunnableLambda로 LLM 응답을 구조화된 딕셔너리로 파싱하기
# ---------------------------------------------------


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **`PromptTemplate`**: `{변수명}` 플레이스홀더로 재사용 가능한 프롬프트를 만들어요
- **`StrOutputParser`**: `AIMessage`에서 텍스트 문자열만 추출해요
- **`RunnableLambda`**: 일반 Python 함수를 파이프에 연결할 수 있는 Runnable(러너블)로 변환해요
- **파이프 연산자(`|`)**: 여러 Runnable을 연결해 데이터 흐름을 직관적으로 표현해요
- **다단계 체인**: `RunnableLambda`로 체인 간 데이터 형식을 변환해 여러 LLM 호출을 연결할 수 있어요

## 다음 노트북 예고

다음 `03_LCEL_basic.ipynb`에서는 LangChain Expression Language(LCEL)의 표준 인터페이스인 `invoke`, `batch`, `stream`과 비동기 메서드를 더 깊이 다뤄요. 체인을 다양한 방식으로 실행하는 법을 알아볼게요.